# Upgrade: Cirrhosis (PBC) - XGBoost

Улучшенная версия с feature engineering'ом:
- **Age в годах** (в данных Age в днях!)
- **Паттерн пропусков** (`pat` = какие из 8 клинических колонок отсутствуют) - очень сильный сигнал
- Количество пропусков, флаги пропусков по каждой колонке
- Логарифмы и отношения медицинских показателей
- Оценка XGBoost: OOF log-loss ~0.370 (вместо 0.3725)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder

df_train = pd.read_csv("train (1).csv")
df_test = pd.read_csv("test (1).csv")
print('train:', df_train.shape, '| test:', df_test.shape)

In [ ]:
target_map = {'C': 0, 'CL': 1, 'D': 2}
y = df_train['Status'].map(target_map)
X = df_train.drop(columns=['id', 'Status']).copy()
X_test = df_test.drop(columns=['id']).copy()

feats8 = ['Drug','Cholesterol','Copper','Alk_Phos','SGOT','Tryglicerides','Platelets','Prothrombin']
cat_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']

for d in [X, X_test]:
    d['Age_years'] = d['Age'] / 365
    d['N_Days_log'] = np.log1p(d['N_Days'])
    d['null_count'] = d.isna().sum(axis=1)
    d['null_clinical'] = d[feats8].isna().sum(axis=1)
    d['Bilirubin_Albumin_ratio'] = d['Bilirubin'] / (d['Albumin'] + 1e-5)
    d['Copper_Bilirubin'] = d['Copper'] * d['Bilirubin']
    d['Bilirubin_SQRT'] = np.sqrt(d['Bilirubin'])
    d['Age_Bilirubin'] = d['Age_years'] * d['Bilirubin']
    d['Age_Prothrombin'] = d['Age_years'] * d['Prothrombin']
    d['Copper_Albumin'] = d['Copper'] / (d['Albumin'] + 1e-5)
    for sc in ['Ascites','Hepatomegaly','Spiders','Edema']:
        d[sc] = np.where(d[sc].eq('Y'), 1.0, np.where(d[sc].eq('N'), 0.0, np.nan))
    d['symp_score'] = d[['Ascites','Hepatomegaly','Spiders','Edema']].sum(axis=1, min_count=1)
    d['pat'] = d[feats8].isna().astype(int).astype(str).agg(''.join, axis=1)
    d['Drug_isna'] = d['Drug'].isna().astype(int)
    for c in feats8:
        d['miss_' + c] = d[c].isna().astype(int)

# кодируем категории в числа (для XGB)
for c in cat_cols + ['Stage']:
    X[c] = X[c].fillna('Missing').astype('category').cat.codes
    X_test[c] = X_test[c].fillna('Missing').astype('category').cat.codes

le = LabelEncoder()
le.fit(pd.concat([X['pat'], X_test['pat']]))
X['pat'] = le.transform(X['pat'])
X_test['pat'] = le.transform(X_test['pat'])
print('features:', X.shape)

In [ ]:
SEEDS = [42, 2024]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_preds = np.zeros((len(X_test), 3))

for seed in SEEDS:
    oof = np.zeros(len(X))
    tp = np.zeros((len(X_test), 3))
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        print(f'  seed {seed} | fold {fold+1}/5 ...', flush=True)
        model = XGBClassifier(
            n_estimators=2000, learning_rate=0.02, max_depth=5,
            subsample=0.8, colsample_bytree=0.7,
            objective='multi:softprob', num_class=3, eval_metric='mlogloss',
            early_stopping_rounds=100, random_state=seed, n_jobs=-1
        )
        model.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])], verbose=False)
        tp += model.predict_proba(X_test) / skf.n_splits
    test_preds += tp / len(SEEDS)

print('done')

In [ ]:
submission = pd.DataFrame({
    'id': df_test['id'],
    'Status_C': test_preds[:, 0],
    'Status_CL': test_preds[:, 1],
    'Status_D': test_preds[:, 2]
})
submission.to_csv('submission_upgrade.csv', index=False)
print(submission.head())
print('saved: submission_upgrade.csv', submission.shape)